In [1]:
# 1. Install Dependencies
!pip install -q langgraph langchain langchain-openai langchain-community pydantic pytest torch 'transformers>=4.51.0,<5' 'accelerate>=1.6.0' 'pypdf>=5.4.0,<6'


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
# 2. Clone a fresh copy of the project on every run
import getpass
import os
import shutil
from pathlib import Path

repo_name = "smart-wallet-optimizer"
repo_url = f"https://github.com/cloudbyte/{repo_name}.git"
current_dir = Path.cwd()
clone_parent = current_dir.parent if current_dir.name == repo_name else current_dir
repo_path = clone_parent / repo_name

if repo_path.exists():
    shutil.rmtree(repo_path)

print(f"Cloning a fresh copy of {repo_name}...")
clone_res = !git clone {repo_url} {repo_path} 2>&1
if any("fatal:" in line for line in clone_res):
    print("🔐 Private repository detected. Please enter your GitHub Personal Access Token (PAT):")
    token = getpass.getpass("GitHub PAT: ").strip()
    if token:
        auth_url = f"https://{token}@github.com/cloudbyte/{repo_name}.git"
        clone_res = !git clone {auth_url} {repo_path} 2>&1

if not repo_path.exists():
    raise RuntimeError("Git clone failed. Check the repository URL and your access permissions.")

os.chdir(repo_path)
print(f"Using fresh clone at {repo_path}")


In [ ]:
# 3. Run Tests
!PYTHONPATH=. pytest


In [ ]:
# 4. Run the Agent
import torch
import os

if torch.cuda.is_available():
    os.environ["WALLET_AGENT_MODEL"] = "Qwen/Qwen3-4B"
    os.environ["WALLET_AGENT_DEVICE"] = "auto"
    print(f"GPU detected ({torch.cuda.get_device_name(0)}); using Qwen3-4B.")
else:
    os.environ["WALLET_AGENT_MODEL"] = "Qwen/Qwen3-0.6B"
    os.environ["WALLET_AGENT_DEVICE"] = "cpu"
    print("No GPU detected; using Qwen3-0.6B on CPU. For faster inference, select a GPU runtime.")

from src.main import main
main()
